## make the dataset for training growth rate

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import glob
import h5py

import sys
sys.path.append('../../../src')
from viz import show_images
from PlumeDataset import plume_dataset
from AutoAlign import align_plumes
from Velocity import VelocityCalculator
from PlumeMetrics import PlumeMetrics

### load growth rate

In [21]:
df_condition = pd.read_excel('../../../../../OneDrive - Drexel University/PLD_SrRuO3/SrRuO3 growth/Sample log.xlsx', 
                             sheet_name='SRO', engine='openpyxl')
df_condition = df_condition.loc[16:33, ['Growth', 'Pressure (mTorr)', 'Fluence (J/cm2)', 'Growth rate (Å/pulse)', 'Pulse number', 'Growth rate (nm/min)']]
df_condition.replace('2 to 1.73', (2+1.73)/2, inplace=True)
df_condition = df_condition.dropna(subset=['Growth rate (Å/pulse)', 'Growth rate (nm/min)'])
df_condition['Growth'] = df_condition['Growth'].str.replace('LYW_YCG', 'YG')
df_condition

,Growth,Pressure (mTorr),Fluence (J/cm2),Growth rate (Å/pulse),Pulse number,Growth rate (nm/min)
16,YG045,75.0,1.692857,0.026700,10000.0,1.60000
17,YG046,75.0,1.865000,0.030875,8000.0,1.80000
18,YG047,75.0,2.050000,0.112000,6000.0,6.70000
22,YG051,75.0,2.050000,0.098033,6000.0,2.94100
24,YG053,75.0,1.600000,0.056600,7000.0,1.69700
27,YG056,100.0,1.600000,0.052600,7000.0,1.57700
28,YG057,75.0,1.230000,0.047286,7000.0,1.41857
29,YG058,125.0,1.600000,0.054000,12000.0,1.62500
30,YG059,50.0,1.600000,0.057600,10000.0,1.72800
31,YG060,75.0,0.800000,0.019000,10000.0,0.57000


In [22]:
df_condition['Growth']

16    YG045
17    YG046
18    YG047
22    YG051
24    YG053
27    YG056
28    YG057
29    YG058
30    YG059
31    YG060
32    YG061
33    YG062
Name: Growth, dtype: object

### load plumes

In [23]:
growth_names = list(df_condition['Growth'].unique())
print(growth_names)

growth_name_dict = {g: i for i, g in enumerate(growth_names)}
growth_name_dict

['YG045', 'YG046', 'YG047', 'YG051', 'YG053', 'YG056', 'YG057', 'YG058', 'YG059', 'YG060', 'YG061', 'YG062']


{'YG045': 0,
 'YG046': 1,
 'YG047': 2,
 'YG051': 3,
 'YG053': 4,
 'YG056': 5,
 'YG057': 6,
 'YG058': 7,
 'YG059': 8,
 'YG060': 9,
 'YG061': 10,
 'YG062': 11}

In [24]:
files = []
for key in growth_names:
    files.append(glob.glob(f'../../../datasets/SRO_STO_Drexel/*{key}*.h5')[0])
files

['../../../datasets/SRO_STO_Drexel\\YG045_YichenGuo_JulianGoddy_05092024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG046_YichenGuo_05162024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG047_YichenGuo_05172024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG051_YichenGuo_06122024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG053_YichenGuo_06172024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG056_YichenGuo_06282024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG057_YichenGuo_06282024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG058_YichenGuo_06292024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG059_YichenGuo_07012024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG060_YichenGuo_07032024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG061_YichenGuo_07042024.h5',
 '../../../datasets/SRO_STO_Drexel\\YG062_YichenGuo_07052024.h5']

In [25]:
length = 0
for file in files:
    print(file)
    plume_ds = plume_dataset(file_path=file, group_name='PLD_Plumes')
    keys = plume_ds.dataset_names()
    plumes = plume_ds.load_plumes('1-SrRuO3')
    length += len(plumes)
    print(len(plumes))

../../../datasets/SRO_STO_Drexel\YG045_YichenGuo_JulianGoddy_05092024.h5


KeyboardInterrupt: 

In [ ]:
plumes.shape, plumes.dtype, np.min(plumes), np.max(plumes), length

In [ ]:
with h5py.File('../../../datasets/growth_rate_regression_ds.h5', 'w') as f:
    f.create_dataset('plumes', shape=(length, 128, 250, 400), dtype=np.uint8)
    f.create_dataset('growth_rate(angstrom_per_pulse)', shape=(length, 1), dtype=np.float32)
    f.create_dataset('growth_rate(nm_per_min)', shape=(length, 1), dtype=np.float32)
    f.create_dataset('growth_name', shape=(length, 1), dtype=np.uint8)

    index = 0
    for growth, file in zip(growth_names, files):
        print(file)
        plume_ds = plume_dataset(file_path=file, group_name='PLD_Plumes')
        plumes = plume_ds.load_plumes('1-SrRuO3')
        f['plumes'][index:index+len(plumes)] = plumes
        print(len(plumes))

        f['growth_rate(angstrom_per_pulse)'][index:index+len(plumes)] = df_condition[df_condition['Growth'] == growth]['Growth rate (Å/pulse)'].values[0]
        f['growth_rate(nm_per_min)'][index:index+len(plumes)] = df_condition[df_condition['Growth'] == growth]['Growth rate (nm/min)'].values[0]
        f['growth_name'][index:index+len(plumes)] = growth_name_dict[growth]

        index += len(plumes)

## cnn + transformer to handle spatial and temporal information

### draft scripts

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet34

class VideoRegressionModel(nn.Module):
    def __init__(self, num_frames, num_channels, patch_size, hidden_dim, num_layers, num_heads, mlp_dim, output_dim):
        super(VideoRegressionModel, self).__init__()
        self.num_frames = num_frames
        self.num_channels = num_channels
        self.patch_size = patch_size
        
        # ResNet34 as the CNN for feature extraction
        self.cnn = resnet34()
        self.cnn.conv1 = nn.Conv2d(num_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.cnn.fc = nn.Identity()  # Remove the last fully connected layer
        
        # Vision Transformer for temporal modeling
        self.transformer = nn.Transformer(d_model=hidden_dim, nhead=num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers, 
                                          dim_feedforward=mlp_dim, dropout=0.1, activation='relu', batch_first=True)

        # Regression head
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = x.unsqueeze(2)  # increase dimension for grayscale channel
        # print(x.shape)

        # Extract features using ResNet34
        features = []
        for i in range(self.num_frames):
            frame_features = self.cnn(x[:, i])
            features.append(frame_features)
        features = torch.stack(features, dim=1)
        # print(features.shape)

        # Pass features through the Transformer
        outputs = self.transformer(features, features)
        # print(transformer_output.shape)
        
        # Take the mean of the Transformer output along the time dimension, act as pooling layer
        outputs = outputs.mean(dim=1) 
        # print(transformer_output.shape)
        
        # Pass the Transformer output through the regression head
        outputs = self.fc(outputs)
        # print(output.shape)
        return outputs
    
model = VideoRegressionModel(num_frames=128, num_channels=1, patch_size=16, hidden_dim=512, 
                             num_layers=4, num_heads=8, mlp_dim=2048, output_dim=1)
input_data = torch.randn(2, 128, 250, 400)
output = model(input_data)
print(output.shape)  # Should be (batch_size, output_dim)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader

import sys
sys.path.append('../../../../Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/src/utils')
sys.path.append('../../../../Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/src/benchmark')
from train_functions import train_epochs
from utils import split_train_valid, viz_dataloader
from viz import show_images

In [ ]:
import h5py
with h5py.File('../../../datasets/growth_rate_regression_ds.h5', 'r') as h5:
    print(h5.keys())
    for key in h5.keys():
        print(key, h5[key].shape, h5[key].dtype)
    images = np.array(h5['plumes'][:10, :32])
    labels = np.array(h5['growth_rate(angstrom_per_pulse)'][:10])
show_images(images[0])
print(labels)

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

import sys
sys.path.append('../../../../Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/src/utils')
sys.path.append('../../../../Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/src/benchmark')
from train_functions import train_epochs
from utils import split_train_valid, viz_dataloader
from viz import show_images

class hdf5_dataset(Dataset):
    
    def __init__(self, file_path, folder=None, transform=None, data_key='data', label_key='labels'):
        self.file_path = file_path
        self.folder = folder
        self.transform = transform
        self.hf = None
        self.data_key = data_key
        self.label_key = label_key

    def __len__(self):
        with h5py.File(self.file_path, 'r') as f:
            if self.folder:
                self.len = len(f[self.folder][self.label_key])
            else:
                self.len = len(f[self.label_key])
        return self.len
    
    def __getitem__(self, idx):
        if self.hf is None:
            self.hf = h5py.File(self.file_path, 'r')
        
        if self.folder:
            video = np.array(self.hf[self.folder][self.data_key][idx])
            label = np.array(self.hf[self.folder][self.label_key][idx])
        else:
            video = np.array(self.hf[self.data_key][idx])
            label = np.array(self.hf[self.label_key][idx])
            
        # Convert numpy array to a list of PIL Images
        if video.dtype != np.uint8:
            video = (video * 255).astype(np.uint8)
        
        video_frames = []
        for frame in video:
            if frame.ndim == 2:  # If it's a grayscale image
                frame = Image.fromarray(frame, mode='L')
            else:
                frame = Image.fromarray(frame)
            video_frames.append(frame)
        
        if self.transform:
            video_frames = [self.transform(frame) for frame in video_frames]
            video_frames = torch.stack(video_frames)
            # Remove the following line to keep the shape as [128, 250, 400]
            # if video_frames.ndim == 3:
            #     video_frames = video_frames.unsqueeze(1)  # expand dimension for single channel video
            video_frames = video_frames.squeeze()  # expand dimension for single channel video

        return video_frames, torch.tensor(label)

In [ ]:
whole_ds = hdf5_dataset('../../../datasets/growth_rate_regression_ds.h5', transform=transforms.ToTensor(),
                        data_key='plumes', label_key='growth_rate(angstrom_per_pulse)')
image, label = whole_ds[0]
print(image.shape, label)

bs = 2
num_workers = 0

# imagenet
whole_ds = hdf5_dataset('../../../datasets/growth_rate_regression_ds.h5', transform=transforms.ToTensor(),
                        data_key='plumes', label_key='growth_rate(angstrom_per_pulse)')
train_ds, valid_ds = split_train_valid(whole_ds, 0.8)
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=num_workers)
valid_dl = DataLoader(valid_ds, batch_size=bs, shuffle=False, num_workers=num_workers)

batch = next(iter(train_dl))
images, labels = batch
print(images.shape, labels.shape)

model = VideoRegressionModel(num_frames=128, num_channels=1, patch_size=16, hidden_dim=512, 
                             num_layers=4, num_heads=8, mlp_dim=2048, output_dim=1)
output = model(images)
print(output.shape)  # Should be (batch_size, output_dim)

In [ ]:
loss_func = nn.MSELoss()
                   
loss = loss_func(output, labels)
print(loss.item())

In [ ]:
loss

### packed function for training

In [1]:
import h5py
import os
import time
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet34
from torchvision import datasets, transforms, models
import matplotlib.image as image

import sys
sys.path.append('../../../../Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/src/utils')
sys.path.append('../../../../Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/src/benchmark')
from utils import split_train_valid, viz_dataloader
from viz import show_images

class hdf5_dataset(Dataset):
    
    def __init__(self, file_path, folder=None, transform=None, data_key='data', label_key='labels'):
        self.file_path = file_path
        self.folder = folder
        self.transform = transform
        self.hf = None
        self.data_key = data_key
        self.label_key = label_key

    def __len__(self):
        with h5py.File(self.file_path, 'r') as f:
            if self.folder:
                self.len = len(f[self.folder][self.label_key])
            else:
                self.len = len(f[self.label_key])
        return self.len
    
    def __getitem__(self, idx):
        if self.hf is None:
            self.hf = h5py.File(self.file_path, 'r')
        
        if self.folder:
            video = np.array(self.hf[self.folder][self.data_key][idx])
            label = np.array(self.hf[self.folder][self.label_key][idx])
        else:
            video = np.array(self.hf[self.data_key][idx])
            label = np.array(self.hf[self.label_key][idx])
            
        # Convert numpy array to a list of PIL Images
        if video.dtype != np.uint8:
            video = (video * 255).astype(np.uint8)
        
        video_frames = []
        for frame in video:
            if frame.ndim == 2:  # If it's a grayscale image
                frame = Image.fromarray(frame, mode='L')
            else:
                frame = Image.fromarray(frame)
            video_frames.append(frame)
        
        if self.transform:
            video_frames = [self.transform(frame) for frame in video_frames]
            video_frames = torch.stack(video_frames)
            # Remove the following line to keep the shape as [128, 250, 400]
            # if video_frames.ndim == 3:
            #     video_frames = video_frames.unsqueeze(1)  # expand dimension for single channel video
            video_frames = video_frames.squeeze()  # expand dimension for single channel video

        return video_frames, torch.tensor(label)

class VideoRegressionModel(nn.Module):
    def __init__(self, num_frames, num_channels, patch_size, hidden_dim, num_layers, num_heads, mlp_dim, output_dim):
        super(VideoRegressionModel, self).__init__()
        self.num_frames = num_frames
        self.num_channels = num_channels
        self.patch_size = patch_size
        
        # ResNet34 as the CNN for feature extraction
        self.cnn = resnet34()
        self.cnn.conv1 = nn.Conv2d(num_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.cnn.fc = nn.Identity()  # Remove the last fully connected layer
        
        # Vision Transformer for temporal modeling
        self.transformer = nn.Transformer(d_model=hidden_dim, nhead=num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers, 
                                          dim_feedforward=mlp_dim, dropout=0.1, activation='relu', batch_first=True)

        # Regression head
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = x.unsqueeze(2)  # increase dimension for grayscale channel
        # print(x.shape)

        # Extract features using ResNet34
        features = []
        for i in range(self.num_frames):
            frame_features = self.cnn(x[:, i])
            features.append(frame_features)
        features = torch.stack(features, dim=1)

        # Pass features through the Transformer
        outputs = self.transformer(features, features)
        
        # Take the mean of the Transformer output along the time dimension, act as pooling layer
        outputs = outputs.mean(dim=1) 
        
        # Pass the Transformer output through the regression head
        outputs = self.fc(outputs)
        return outputs
    

def train_epochs(model, loss_func, optimizer, device, train_dl, valid_dl_list, valid_name_list,
                 epochs, start=0, scheduler=None, valid_every_epochs=1, accuracy=True, model_dir=None, tracking=False):
    
    if isinstance(valid_every_epochs, int) and valid_dl_list != []:
        valid_every_epochs = [valid_every_epochs]*len(valid_dl_list)
        
    elif isinstance(valid_every_epochs, list) and valid_dl_list == []:
        if len(valid_every_epochs) != len(valid_dl_list):
            raise ValueError("The length of valid_every_epochs should match the number of cross-validation datasets.")
    
    # make directory for the model
    if model_dir and not os.path.isdir(model_dir): 
        os.mkdir(model_dir)

    history = {'epoch':[], 'train_loss':[], 'valid_loss':[], 'train_acc':[], 'valid_acc':[]}
    if valid_dl_list != []:
        for cv_name in valid_name_list:
            history[cv_name+'_loss'] = []
        
    for epoch_idx in range(start, epochs+start):
                
        print("Epoch: {}/{}".format(epoch_idx+1, epochs+start))
        
        avg_train_loss = train(model, loss_func, optimizer, device, train_dl, 
                              scheduler=scheduler, tracking=tracking)
        print(f"Training: Loss: {avg_train_loss:.4f}")
        
        metadata = {'epoch': epoch_idx, 'train_loss': avg_train_loss}

        if valid_dl_list != []:
            for i, (cv_dl, cv_name) in enumerate(zip(valid_dl_list, valid_name_list)):
                if (epoch_idx+1) % valid_every_epochs[i] == 0: # the first element is for the validation set
                    avg_cv_loss = valid(model, loss_func, device, cv_dl, task_label=cv_name, tracking=tracking)
                    metadata[cv_name+'_loss'] = avg_cv_loss
                    print(f"{cv_name}: Loss: {avg_cv_loss:.4f}")

        if model_dir != None:
            torch.save(model.cpu(), os.path.join(model_dir, 'epoch-{}.pt'.format(epoch_idx+1)))

        # record the epoch loss and accuracy:
        for key, value in metadata.items():
            history[key].append(value)
                
    return history


def train(model, loss_func, optimizer, device, train_dl, scheduler=None, tracking=False):

    train_data_size = len(train_dl.dataset)
    start_time = time.time()

    # Set to training mode
    model.train()

    # Loss and Accuracy within the epoch
    train_loss = 0.0

    for i, batch in enumerate(tqdm(train_dl)):
        inputs = batch[0].to(device).float()
        labels = batch[1].to(device).float()
        model = model.to(device)

        # Clean existing gradients
        optimizer.zero_grad()

        # Forward pass - compute outputs on input data using the model
        outputs = model(inputs)

        # Compute loss
        loss = loss_func(outputs, labels) 

        # Compute the total loss for the batch and add it to train_loss
        train_loss += loss.item() * inputs.size(0)
        
        # Backpropagate the gradients
        loss.backward()

        # Update the parameters
        optimizer.step()
        if scheduler:
            scheduler.step()

    # Find average training loss and training accuracy
    avg_train_loss = train_loss/train_data_size 

    return avg_train_loss


def valid(model, loss_func, device, valid_dl, task_label='valid', tracking=False):

    valid_data_size = len(valid_dl.dataset)

    # Loss and Accuracy within the epoch
    valid_loss = 0.0
    valid_acc = 0.0
    
    start_time = time.time()
        
    # Validation - No gradient tracking needed
    with torch.no_grad():

        # Set to evaluation mode
        model.eval()

        # Validation loop
        
        for j, batch in enumerate(tqdm(valid_dl)):
            
            inputs = batch[0].float().to(device)
            labels = batch[1].long().to(device)

            model = model.to(device)

            # Forward pass - compute outputs on input data using the model
            outputs = model(inputs)
            
            # Compute loss
            loss = loss_func(outputs, labels) 
            
            # Compute the total loss for the batch and add it to valid_loss
            valid_loss += loss.item() * inputs.size(0)

    # Find average training loss and training accuracy
    avg_valid_loss = valid_loss/valid_data_size 

    return avg_valid_loss

In [2]:
bs = 2
num_workers = 0

# imagenet
whole_ds = hdf5_dataset('../../../datasets/growth_rate_regression_ds.h5', transform=transforms.ToTensor(),
                        data_key='plumes', label_key='growth_rate(angstrom_per_pulse)')
train_ds, valid_ds = split_train_valid(whole_ds, 0.8)
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=num_workers)
valid_dl = DataLoader(valid_ds, batch_size=bs, shuffle=False, num_workers=num_workers)

batch = next(iter(train_dl))
images, labels = batch
print(images.shape, labels.shape)

model = VideoRegressionModel(num_frames=128, num_channels=1, patch_size=16, hidden_dim=512, 
                             num_layers=4, num_heads=8, mlp_dim=2048, output_dim=1)
output = model(images)
print(output.shape)  # Should be (batch_size, output_dim)

torch.Size([2, 128, 250, 400]) torch.Size([2, 1])
torch.Size([2, 1])


In [3]:
device = torch.device('cuda:0')
lr = 1e-3
start = 0
epochs = 20

loss_func = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, steps_per_epoch=len(train_dl))
history = train_epochs(model, loss_func, optimizer, device, train_dl, valid_dl_list=[valid_dl], valid_name_list=['validation'],
                       epochs=epochs, start=start, scheduler=scheduler, valid_every_epochs=1, tracking=False)

Epoch: 1/20


  0%|          | 0/1212 [00:00<?, ?it/s]c:\Users\yig319\Anaconda3\envs\plume\Lib\site-packages\torch\nn\functional.py:5504: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\yig319\Anaconda3\envs\plume\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
  2%|▏         | 27/1212 [00:57<39:03,  1.98s/it] 